# JPEG Double-Compression Detector — Local Jupyter Setup
**GPU-accelerated training with a pre-downloaded local dataset.**

In [ ]:
pip install jpegio torch torchvision numpy scipy tqdm matplotlib opencv-python

In [ ]:
import os, torch, cv2, numpy as np
import torch.nn as nn
import torch.nn.functional as F
import jpegio as jio
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, Subset
from tqdm import tqdm
from scipy.ndimage import gaussian_filter

# ── GPU setup ──────────────────────────────────────────────────────────────
assert torch.cuda.is_available(), "No CUDA GPU found! Check your drivers."
DEVICE = torch.device("cuda")
print(f"Using Device : {DEVICE}")
print(f"GPU Name     : {torch.cuda.get_device_name(0)}")
print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

torch.backends.cudnn.benchmark = True
torch.manual_seed(42)
np.random.seed(42)

## Configuration
Set `DATASET_DIR` to the root of your **already-extracted** dataset.
Expected layout:
```
DATASET_DIR/
  train/
    single/   *.jpg
    double/   *.jpg
  val/
    single/   *.jpg
    double/   *.jpg
  test/
    single/   *.jpg
    double/   *.jpg
```

### ✏️  Split-size knobs
Set any limit to `None` to use **all** available files in that split folder.
The limit applies **per class** (single / double), so the dataset stays balanced.

| Variable | Meaning |
|---|---|
| `TRAIN_LIMIT` | Max images per class used for training |
| `VAL_LIMIT`   | Max images per class used for validation |
| `TEST_LIMIT`  | Max images per class used for testing |

In [ ]:
# ── Paths ───────────────────────────────────────────────────────────────────
DATASET_DIR    = '/data/user12/team13/anurag/dataset'
CHECKPOINT_DIR = '/data/user12/team13/anurag/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Split-size knobs  (None = use everything in the folder) ─────────────────
# Values are per-class (single AND double each get this many samples).
# Paper defaults: train=171065, val=57022, test=57022  (30/10/10 split)
TRAIN_LIMIT = 171065   # e.g. 10000, 50000, 171065, or None
VAL_LIMIT   =  57022   # e.g.  5000, 20000,  57022, or None
TEST_LIMIT  =  57022   # e.g.  5000, 20000,  57022, or None

# Quick sanity check — shows total files available
print("Available files in dataset:")
for split in ['train', 'val', 'test']:
    for cls in ['single', 'double']:
        p = os.path.join(DATASET_DIR, split, cls)
        n = len(os.listdir(p)) if os.path.exists(p) else 0
        print(f"  {split}/{cls}: {n:,} files")

## Model — DenseNet-121 (JPEG DCT input)

In [ ]:
class DenseLayer(nn.Module):
    def __init__(self, in_ch, k):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm2d(in_ch), nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, 4*k, 1, bias=False),
            nn.BatchNorm2d(4*k), nn.ReLU(inplace=True),
            nn.Conv2d(4*k, k, 3, padding=1, bias=False)
        )
    def forward(self, x):
        return torch.cat([x, self.net(x)], 1)

class DenseNet121_JPEG(nn.Module):
    def __init__(self, num_classes=2, k=32):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(2, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1)
        )
        in_ch, config = 64, [6, 12, 24, 16]
        self.blocks = nn.ModuleList()
        for i, nl in enumerate(config):
            block = nn.Sequential(*[DenseLayer(in_ch + j*k, k) for j in range(nl)])
            self.blocks.append(block)
            in_ch += nl * k
            if i < 3:
                trans = nn.Sequential(
                    nn.BatchNorm2d(in_ch), nn.ReLU(inplace=True),
                    nn.Conv2d(in_ch, in_ch//2, 1, bias=False),
                    nn.AvgPool2d(2, stride=2)
                )
                self.blocks.append(trans)
                in_ch //= 2
        self.final_bn = nn.BatchNorm2d(in_ch)
        self.classifier = nn.Linear(in_ch, num_classes)

    def forward(self, x):
        x = self.stem(x)
        for b in self.blocks: x = b(x)
        x = F.relu(self.final_bn(x))
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.classifier(x)

## Dataset & Feature Extraction

In [ ]:
def extract_features(path, patch_coords=None):
    """Extract 64×(2b+1)×2 feature tensor from a JPEG file."""
    try:
        jpg = jio.read(path)
        if patch_coords:
            y, x = patch_coords
            dct = jpg.coef_arrays[0][y:y+256, x:x+256]
        else:
            dct = jpg.coef_arrays[0][:256, :256]
        q = jpg.quant_tables[0].reshape(8,8) if jpg.quant_tables[0].ndim==1 else jpg.quant_tables[0]
        if dct.shape != (256, 256): return None

        b = 80
        bins = np.arange(-b, b+2) - 0.5
        H = np.zeros((64, 2*b+1), dtype=np.float32)
        Q = np.zeros((64, 2*b+1), dtype=np.float32)
        for idx, (u, v) in enumerate((u,v) for u in range(8) for v in range(8)):
            c = np.clip(dct[u::8, v::8].flatten(), -b, b)
            H[idx], _ = np.histogram(c, bins=bins)
            Q[idx] = float(q[u,v])
        X = np.stack([H, Q], axis=-1)
        X /= (X.max() + 1e-6)
        return X
    except:
        return None


class JPEGDataset(Dataset):
    """
    Loads JPEG patches from root_dir/single/ and root_dir/double/.

    Parameters
    ----------
    root_dir      : path that contains 'single/' and 'double/' sub-folders
    max_per_class : if given, randomly sample this many files from each class.
                    None → use all files (original behaviour).
    seed          : random seed used for sampling (for reproducibility).
    """
    def __init__(self, root_dir, max_per_class=None, seed=42):
        rng = np.random.default_rng(seed)
        self.file_list = []
        for label, cls in enumerate(['single', 'double']):
            path = os.path.join(root_dir, cls)
            if not os.path.exists(path):
                continue
            files = sorted([f for f in os.listdir(path) if f.lower().endswith('.jpg')])
            if max_per_class is not None and max_per_class < len(files):
                # Random subsample, reproducible
                idx = rng.choice(len(files), size=max_per_class, replace=False)
                files = [files[i] for i in sorted(idx)]
            for f in files:
                self.file_list.append((os.path.join(path, f), label))

    def __len__(self): return len(self.file_list)

    def __getitem__(self, idx):
        path, label = self.file_list[idx]
        X = extract_features(path)
        if X is None: return torch.zeros((2, 64, 161)), label
        return torch.tensor(X).permute(2, 0, 1), label


def dataset_summary(ds, name):
    """Print class counts for a JPEGDataset."""
    singles = sum(1 for _, lbl in ds.file_list if lbl == 0)
    doubles = sum(1 for _, lbl in ds.file_list if lbl == 1)
    print(f"  {name:6s}: {len(ds):>8,} total  "
          f"(single={singles:,}  double={doubles:,})")

## DataLoaders & Training

In [ ]:
# ── DataLoaders ─────────────────────────────────────────────────────────────
NUM_WORKERS = 0
BATCH_SIZE  = 64
PIN_MEMORY  = True

# Pass the limit knobs defined in the Config cell
train_dataset = JPEGDataset(os.path.join(DATASET_DIR, 'train'), max_per_class=TRAIN_LIMIT)
val_dataset   = JPEGDataset(os.path.join(DATASET_DIR, 'val'),   max_per_class=VAL_LIMIT)
test_dataset  = JPEGDataset(os.path.join(DATASET_DIR, 'test'),  max_per_class=TEST_LIMIT)

print("\nDataset sizes (after applying limits):")
dataset_summary(train_dataset, 'Train')
dataset_summary(val_dataset,   'Val')
dataset_summary(test_dataset,  'Test')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

# ── Model & Optimizer ───────────────────────────────────────────────────────
model     = DenseNet121_JPEG().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
scaler    = torch.amp.GradScaler('cuda')

EPOCHS      = 30
LR_DROP_EP  = 21       # epoch index at which LR drops (0-based)
LR_DROP_VAL = 5e-4
best_val_acc = 0.0

print(f"\nTraining for {EPOCHS} epochs on {DEVICE}...\n")

for epoch in range(EPOCHS):
    if epoch == LR_DROP_EP:
        for pg in optimizer.param_groups: pg['lr'] = LR_DROP_VAL
        print(f"  >> LR dropped to {LR_DROP_VAL}")

    # ── TRAINING ──────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    tp, tn, fp, fn = 0, 0, 0, 0

    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        inputs, labels = inputs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda'):
            outputs = model(inputs)
            loss    = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        tp += ((preds == 1) & (labels == 1)).sum().item()
        tn += ((preds == 0) & (labels == 0)).sum().item()
        fp += ((preds == 1) & (labels == 0)).sum().item()
        fn += ((preds == 0) & (labels == 1)).sum().item()

    train_acc = (tp + tn) / (tp + tn + fp + fn + 1e-7)

    # ── VALIDATION ────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    v_tp, v_tn, v_fp, v_fn = 0, 0, 0, 0

    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val  ]"):
            inputs, labels = inputs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            with torch.amp.autocast('cuda'):
                outputs = model(inputs)
                loss    = criterion(outputs, labels)
            val_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            v_tp += ((preds == 1) & (labels == 1)).sum().item()
            v_tn += ((preds == 0) & (labels == 0)).sum().item()
            v_fp += ((preds == 1) & (labels == 0)).sum().item()
            v_fn += ((preds == 0) & (labels == 1)).sum().item()

    val_acc = (v_tp + v_tn) / (v_tp + v_tn + v_fp + v_fn + 1e-7)
    val_tpr = v_tp / (v_tp + v_fn + 1e-7)
    val_tnr = v_tn / (v_tn + v_fp + 1e-7)

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss   = val_loss   / len(val_loader)

    print(f" Epoch {epoch+1:02d}")
    print(f"   Train | Loss: {avg_train_loss:.4f} | Acc: {train_acc*100:.2f}%")
    print(f"   Val   | Loss: {avg_val_loss:.4f} | Acc: {val_acc*100:.2f}% "
          f"| TPR: {val_tpr*100:.2f}% | TNR: {val_tnr*100:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        ckpt_path = os.path.join(CHECKPOINT_DIR, 'best_model.pth')
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': best_val_acc,
        }, ckpt_path)
        print(f"  Best model saved → {ckpt_path}  (Val Acc: {best_val_acc*100:.2f}%)")

print(f"\nTraining complete. Best Val Acc: {best_val_acc*100:.2f}%")

## Locate saved checkpoint

In [ ]:
import glob
pth_files = glob.glob(os.path.join(CHECKPOINT_DIR, '**', '*.pth'), recursive=True)
if pth_files:
    for p in pth_files:
        size_mb = os.path.getsize(p) / 1e6
        print(f"Found: {p}  ({size_mb:.1f} MB)")
else:
    print('No .pth files found in CHECKPOINT_DIR.')

---
## 🧪 Standalone Test Block
Run this cell **independently** at any time — it loads the best saved checkpoint
and evaluates it on the test set.  
You do **not** need to re-run training to use this block.

Metrics reported: **Accuracy**, **TPR** (True Positive Rate / Recall for double),
**TNR** (True Negative Rate / Recall for single), and **per-class counts**.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  STANDALONE TEST BLOCK
#  • Can be run independently of the training cell above.
#  • Loads the best checkpoint and evaluates on the test split.
#  • Adjust CHECKPOINT_PATH and TEST_LIMIT_STANDALONE below if needed.
# ════════════════════════════════════════════════════════════════════════════

import os, torch, numpy as np
import torch.nn as nn
import torch.nn.functional as F
import jpegio as jio
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# ── Settings (edit as needed) ───────────────────────────────────────────────
CHECKPOINT_PATH      = '/data/user12/team13/anurag/checkpoints/best_model.pth'
TEST_DIR             = '/data/user12/team13/anurag/dataset/test'
TEST_LIMIT_STANDALONE = 57022   # per-class limit for testing; None = all
BATCH_SIZE_TEST      = 64
NUM_WORKERS_TEST     = 0

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running test on: {DEVICE}")

# ── Re-define helpers (safe to re-run even if training cell already ran) ────
def extract_features(path, patch_coords=None):
    try:
        jpg = jio.read(path)
        if patch_coords:
            y, x = patch_coords
            dct = jpg.coef_arrays[0][y:y+256, x:x+256]
        else:
            dct = jpg.coef_arrays[0][:256, :256]
        q = jpg.quant_tables[0].reshape(8,8) if jpg.quant_tables[0].ndim==1 else jpg.quant_tables[0]
        if dct.shape != (256, 256): return None
        b = 80
        bins = np.arange(-b, b+2) - 0.5
        H = np.zeros((64, 2*b+1), dtype=np.float32)
        Q = np.zeros((64, 2*b+1), dtype=np.float32)
        for idx, (u, v) in enumerate((u,v) for u in range(8) for v in range(8)):
            c = np.clip(dct[u::8, v::8].flatten(), -b, b)
            H[idx], _ = np.histogram(c, bins=bins)
            Q[idx] = float(q[u,v])
        X = np.stack([H, Q], axis=-1)
        X /= (X.max() + 1e-6)
        return X
    except:
        return None

class JPEGDataset(Dataset):
    def __init__(self, root_dir, max_per_class=None, seed=42):
        rng = np.random.default_rng(seed)
        self.file_list = []
        for label, cls in enumerate(['single', 'double']):
            path = os.path.join(root_dir, cls)
            if not os.path.exists(path): continue
            files = sorted([f for f in os.listdir(path) if f.lower().endswith('.jpg')])
            if max_per_class is not None and max_per_class < len(files):
                idx = rng.choice(len(files), size=max_per_class, replace=False)
                files = [files[i] for i in sorted(idx)]
            for f in files:
                self.file_list.append((os.path.join(path, f), label))
    def __len__(self): return len(self.file_list)
    def __getitem__(self, idx):
        path, label = self.file_list[idx]
        X = extract_features(path)
        if X is None: return torch.zeros((2, 64, 161)), label
        return torch.tensor(X).permute(2, 0, 1), label

class DenseLayer(nn.Module):
    def __init__(self, in_ch, k):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm2d(in_ch), nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, 4*k, 1, bias=False),
            nn.BatchNorm2d(4*k), nn.ReLU(inplace=True),
            nn.Conv2d(4*k, k, 3, padding=1, bias=False)
        )
    def forward(self, x): return torch.cat([x, self.net(x)], 1)

class DenseNet121_JPEG(nn.Module):
    def __init__(self, num_classes=2, k=32):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(2, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1)
        )
        in_ch, config = 64, [6, 12, 24, 16]
        self.blocks = nn.ModuleList()
        for i, nl in enumerate(config):
            block = nn.Sequential(*[DenseLayer(in_ch + j*k, k) for j in range(nl)])
            self.blocks.append(block)
            in_ch += nl * k
            if i < 3:
                trans = nn.Sequential(
                    nn.BatchNorm2d(in_ch), nn.ReLU(inplace=True),
                    nn.Conv2d(in_ch, in_ch//2, 1, bias=False),
                    nn.AvgPool2d(2, stride=2)
                )
                self.blocks.append(trans)
                in_ch //= 2
        self.final_bn   = nn.BatchNorm2d(in_ch)
        self.classifier = nn.Linear(in_ch, num_classes)
    def forward(self, x):
        x = self.stem(x)
        for b in self.blocks: x = b(x)
        x = F.relu(self.final_bn(x))
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.classifier(x)

# ── Load checkpoint ─────────────────────────────────────────────────────────
assert os.path.exists(CHECKPOINT_PATH), f"Checkpoint not found: {CHECKPOINT_PATH}"
ckpt  = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model = DenseNet121_JPEG().to(DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
trained_epoch = ckpt.get('epoch', '?')
saved_val_acc = ckpt.get('val_acc', float('nan'))
print(f"\nLoaded checkpoint from epoch {trained_epoch}  "
      f"(saved val_acc = {saved_val_acc*100:.2f}%)")

# ── Build test loader ───────────────────────────────────────────────────────
test_ds = JPEGDataset(TEST_DIR, max_per_class=TEST_LIMIT_STANDALONE)
singles = sum(1 for _, l in test_ds.file_list if l == 0)
doubles = sum(1 for _, l in test_ds.file_list if l == 1)
print(f"\nTest set: {len(test_ds):,} images  "
      f"(single={singles:,}  double={doubles:,})")

test_loader_sa = DataLoader(
    test_ds, batch_size=BATCH_SIZE_TEST, shuffle=False,
    num_workers=NUM_WORKERS_TEST, pin_memory=(DEVICE.type == 'cuda')
)

# ── Evaluation loop ─────────────────────────────────────────────────────────
tp, tn, fp, fn = 0, 0, 0, 0
test_loss = 0.0
criterion = nn.CrossEntropyLoss()

with torch.no_grad():
    for inputs, labels in tqdm(test_loader_sa, desc="Testing"):
        inputs = inputs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.amp.autocast('cuda') if DEVICE.type == 'cuda' else torch.no_grad():
            outputs = model(inputs)
            loss    = criterion(outputs, labels)
        test_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        tp += ((preds == 1) & (labels == 1)).sum().item()
        tn += ((preds == 0) & (labels == 0)).sum().item()
        fp += ((preds == 1) & (labels == 0)).sum().item()
        fn += ((preds == 0) & (labels == 1)).sum().item()

# ── Results ─────────────────────────────────────────────────────────────────
total    = tp + tn + fp + fn
accuracy = (tp + tn) / (total + 1e-7) * 100
tpr      = tp / (tp + fn + 1e-7) * 100   # Recall for double-compressed
tnr      = tn / (tn + fp + 1e-7) * 100   # Recall for single-compressed
precision = tp / (tp + fp + 1e-7) * 100
f1       = 2 * precision * tpr / (precision + tpr + 1e-7)
avg_loss = test_loss / len(test_loader_sa)

print("\n" + "="*50)
print("           TEST RESULTS")
print("="*50)
print(f"  Samples tested : {total:,}")
print(f"  Test Loss      : {avg_loss:.4f}")
print(f"  Accuracy       : {accuracy:.2f}%")
print(f"  TPR (double)   : {tpr:.2f}%")
print(f"  TNR (single)   : {tnr:.2f}%")
print(f"  Precision      : {precision:.2f}%")
print(f"  F1-Score       : {f1:.2f}%")
print("─"*50)
print(f"  TP={tp:,}  TN={tn:,}  FP={fp:,}  FN={fn:,}")
print("="*50)